# DCIE × Survival State Mechanics | ARC-AGI-3 official submission

**Research attribution:** Survival State Mechanics (SSM), original theory by **Joshua Matthew Carter** (2026). The accompanying executable is a bounded operationalizion, **not** a proven complete implementation of the original κ-state theory. Supporting FARSEER/Rotor/CausalNavigator code originates in earlier project research; official runtime components originate with the ARC Prize Foundation.

**This embeds two independent trainable PyTorch recursive mixture-of-experts models, real on-action test-time training, contextual survival-morph evidence, multi-step causal counterfactual scoring, FARSEER navigation and causal clicks**. During **official competition rerun**, it joins the supplied `ARC-AGI-3-Agents` framework and plays the games issued by Kaggle's `gateway:8001`; the gateway owns the actual competition scoring and writes the official output. The Save & Run All/commit branch emits only the placeholder required by the official starter, not a benchmark result.

**Limits:** The models start with random weights and train online on official observations; they are not pretrained. GPU initialization is requested; actual allocation can only be confirmed on Kaggle. Real hidden-game quality and official score have **not** been measured here. Neither this notebook nor the SSM model should be presented as a formal mathematical validation of Carter's theory without independent ablations.


In [ ]:
import sys, os, subprocess, pathlib
COMPETITION_ROOT=pathlib.Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
WHEELS=COMPETITION_ROOT/'arc_agi_3_wheels'
FRAMEWORK_SOURCE=COMPETITION_ROOT/'ARC-AGI-3-Agents'
if not WHEELS.exists() or not FRAMEWORK_SOURCE.exists():
    raise FileNotFoundError('Official ARC-AGI-3 Kaggle competition assets missing; attach ARC-AGI-3 competition data')
subprocess.check_call([sys.executable,'-m','pip','install','--no-index','--find-links',str(WHEELS),'arc-agi','python-dotenv'])
import torch
print('torch=',torch.__version__,'GPU available=',torch.cuda.is_available(),'device=',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
import base64,io,zipfile, pathlib,sys,hashlib
RUNTIME=pathlib.Path('/tmp/dcie_arc3_runtime')
RUNTIME.mkdir(parents=True,exist_ok=True)
FARSEER_PAYLOAD='UEsDBBQAAAAIAFx7Yl0bBEtYKwAAAC0AAAAWAAAAcmF6YWVsX2NoaS9fX2luaXRfXy5weVNSUnIsLcnIL8qsSk1RKEpNzi9LLQKzilMTi5IzFIpK80oyc1P1lJSUuABQSwMEFAAAAAgAXHtiXUNA5YijAAAAIgEAABsAAAByYXphZWxfY2hpL2FyYzMvX19pbml0X18ucHlljjEPgjAQhff+iktnYHF2IAzGjbA4GNMgFL2k5Uh7aPz31haj6E2X771776SUZVPl5W6fb8DPZ4veI43g5pHRahjIAV81NMRhu5Mzfe65ZQ0TGewehZRSiMGRhSIRQDuR43QRsjd1xIsnJrwtZd9OjDcdrYeXkqWz2ukeOw6PLCCKFY0DXoRQqjVGKdjCUUAY+Z8js6T8PLHCn5IV/qoK/CSeUEsDBBQAAAAIAFx7Yl3UvFLoYQcAAI0WAAAbAAAAcmF6YWVsX2NoaS9hcmMzL2ZlYXR1cmVzLnB5zVhZj9s2EH7fX8EYKEBtZcXaK4kBL7DIUeShRRGkfTEMh5Zom11dJeXddYr+985wKImy7E0K9KELxDHJuTjzzUGPRqN3spY6V4UytUrYWotcsrUU9U5Lw0SRsiRTyT1L4KtKRQ2b61Kzu09vx3c/fRxfRqPR6OxsrcucLZfrHbItl0zlValrYC/KWtSqLMzZmdvbCrPN1KpZ/mHKgtiTMstkYokb/rflrgDrQpbKP3eSyOp9pYpNQ3FX7FvJxS6v9kwYVlREajeiPsMv7+60FsB09uH93effPr1fvvv4M5uxmyvYSuWaLQWec+uIKcoP2Pi2YZsXVaSKOr5ZTM8Y/D2IbCeBG7aF8Rgj+wl2g3I5a5gCy6PWxBaBQ3P2YsYuGTiUtsxWVHI+WbDZjE0Ot+Pj2xe0TQbhnxbKSPY7krzXutR8BNFykc13pmYryQQrymIs86reMw1eF8VmlwnN5pnYS21CtpVqs61D9qjSertgG63SUWs/RBWvrMwacFNLbs0JIpFlPPg+O5KyqIWCSKMZJIUuZZwWLQFKBe25yFjOJRBvpK40ePQwSKbW/bD4sSSxldhnpUjh7K/WzpH142jKMsgB7rk2CDsaG0mgAR2OxO74JInMMgMkdAzpgzL4OA6iurSiHfHf9lMWSZlKtARTIEoBq4Y780JmZCW0qEttZnwUjkI2mo4C2AYIL+/l3sw+a/Q4CeGjXb0ev+57zuUZ3uTi+oY7dUG0lU+p2kg0x/k1ETsjsuV/4V4D2b4zS10+GjjPxROPQ4aiNGRyyg/QfM4m0eQ6CIj3sdQZusMSzachm449cbBcdKnTirjtqZSZcTD6zmBbnc8Hm0hOBpuOTwfbEmfyAUxbJmVeZVBuU+BDp2wk2A4qXLEYkoVsEvxPQUNHBJpniuUaLKtvrhbfAs7RstOAgQJk6drOZGvuV6lLw71C7lVcp9ldr3xcmkRkskMlaWJjFhMJtJ9dXhxSkR0tEbY+IIRPVUB9g1Th8cQreVVpFHUw6gl687iV2hVILNOWOWjpV8KgMhJ5zq7bg+aec6SAAt9J7nrEy8Y76qtsGSFFPFI46YwbyGU/srgnG7MuXkS5FAUPQH7rtmdlXAxkXHgyfL8+K+bymCmmTv+FJVfHLGlE9AxxIYfxwoVqpQq75PAdJp6KD6o4JGPI3sB/MCxlstjU2xnEHvo+Io4fIq7S5UqsVAamWLSSrrlTects/GgVmV3OiWtbavUVW2MGLLyrhOALKH8v/OKI9TFeBI2jIe6E1VsWUx2E2kpJJzVMdgcS46ktqD2J4xj3fIkuRwYiV6VOpSbHQSdPYCws4B/v5E9QVN977Rnp8Q+DdrJoNV50uD2pjQ48ybG9w+SUYjoGrx3RnZepdZEiCEDmYgWgADmSFm3XkykCjbcWAiJ4U8FeI9ZeR5Owd9rUNYAHHN9cHZ7b2B0/trByrsWAWUubKAWHdGPeR945ei0rN/3tICDQgTJ3HENsh9I6PA6OGlgNDmyUwG1LGO2wQDdOBF1WyUCQTbS2Xrw5SdIk8mkK9OQ3KCCopygcnp7zcdwE5uQo3IDkxDT8ATXBiPVrCUBzU7E3AncDciOn35+bXdeEYWxTEjydlGC5wqQwh7243sEsMadPUGmnsUXIoihyPRneb59IeNp7CX7hTyHbB1+OvP/wbfWgjFplOMjDu6qQWMbwJfh8l1e52Mh2xINEpKJ30PMtldfz2zLtnoMccxTnr4D6MXzDdkxsfmZTrRLJ/caOn9jVASAkLmQwEc0yka9SQf13yvi4rdKwBi9Rt24ikMODBV+S/jgJg+1j6FpL19bRLDjohgS6Yp+AmDoae/3Ab+R4T3uruadlESBAu1t502EXCxpv/bA/g4MFFrM5xeJxqyCo7V07+MKYDa9n8mB73FlLE8zMM9kyLDqKSgKArPfsY54PCFqpEMO8fJDcEngqcOLuLO2sdZL7g47nMZwJiCSqSrj/uuZBj9ZKjkSFVLwX0T4dRq3ASK5oALS0OBy29KEFBM5VB1utLThKHmwBdRD0rW+Ki6eswx/+KNP5uaEBXDTz5VDWUf82nMFR8sZlzi1DYi/zG9dZ7Q5rHJ8dMrWJCv0GXGk5bYmV6RCgiEcPiJ50FMS9bIVCmUOyulzFFUzDgGUYyrhdwVgUMrfvvsBWN8EvQwcmcKuXMx7SsMzgkxLswT6J31xOh2xp2eytoJGgUlocsrfgQwlu0WhvU/85OVAW7u1bz3lyiNmwt+UXNDidMj4IrFgZqwWfD2PvngGAsD2LvbMGxANJSNnf9Yi6O7hwd8ll7wTqQUvIutVkEQzKp+8n8o2NJ33D2beP80NVxByipAZ8rl83GDxpJXWily/ZRftOxIUHoT2+fO2Plda6SXRxDYNnZD9eXXuWIfHT9xIfs6WdDqnQYOLT8tyTG9CzhHePWqJ063PP3iDwrqGKVD51LQjGz9NO4fErfKW9AmmWLWA/MDe28jjGd+Clf+RaXm98ocKQqqSOcIjA3yS4UwOE/wBQSwMEFAAAAAgAXHtiXWshIoodCAAA2hoAACAAAAByYXphZWxfY2hpL2FyYzMvaW50ZXJ2ZW50aW9ucy5weaVZS4/jNhK++1cwfZKmbcXOIjkYcZDBYA6LBTZBNsjFMARaotzc0cMhqe52JvPfU1UkRerR6ZndPnhs1YNVXz1Ypbm7u/vp/F9RmI0SNTfyUbCC95rXTLZGqEfRGtm1mlWdYleueCPgqfxDlOztL+8YL4ia3d3drVaV6hqW51VveiXynMnm2inDeNt2hhOf4ym6uhZW0jO963o8bs1K8XsvLFvJDS9qrrUY2IZHlqPh5sGT9O/K2KfmdpXtxT9/295WK/e97ZvrjXHN2utqtfrpKhQ3nfqPvLQcbWYHZvprLY7g+prNP06rf7ba8LYQ/xK3gXmmJhZYrX4cbE7Auj9Ee/hV9SJd0SP2rpbFh1+5ugizXzH4AyTfskep5bkWoxAwQ1wAZ8mk0cwo3uqacN3I9pEryVuDvgsbDVRWdJ0qJVgl9hPPTkTX3uQ9m3lhNfx4VR1QzI1+laICYQtB/kHcEi3qKmWbH1gEjPUD/5QARS0jrkwHeOh3sO24Pc2f7U7pCDyHV/7+UZYCDrKncGNEczV6jz5BRLb0VFQVpNfkIThyUUJPnlZc1mBT9HS1Qi/FM+BbmLzA+OQWegwg5P8eU4qctpBGIVyzLMtOQyDfdc1VQhi7FopKYCzHEQVgh1BrSNlabKA0oNAKEASh1oRQPvK6xwRtrxnXXCl+s9Zk9Al1g5E/ABVO2H2XkoysrFjWlrJhXx3YPxgob2Sb2Mf6gV9Fyr4Ht5EAZYrqpa5kK42wTGnG6zpJo6ByqQX7DWnvlepUcoeNgKxgTa8NOwvGmVUBKtsNBujGjjW/CaXX7EHIywMg9SRL83BiFyXLO2duwy/oIp173Oxsjo74gUpc1nSbw9Bbep2r7kkDteHPyY4yPFHQUsrEirM3bJttv01Te9JTp+oyd6RByP3exCojfn/2cR+Ln1yhQfvC810fS9CAQtR1So0Tv4FNVk8G6YbWJ5udM+fMiw8XMheNgfBYdWsGJXaoeXMuOaPE2LNkY2lH+g11Q/86PUo0HHCH3ndgH4eAARBPxNc3bTo8RbOAgFZBI7mIJHZqzGZFI06MRGCRFcFN8sforFOKKRd8I4FPDi6X31B3tdTmGDUn+3XSrWxhnU7g2NEi/vSAlTV4HPITgqeMw3EgB2ttgR0ik0ngFDiuAsqFMKS7KJkxDFohkk33KBJiiI6AaC84hn5EDgQn3IHBBa8EmB0tu3agpzJJOmPSGb8il023ETkKBqhC+oiMsW0x3ucOGxNLRlSfOlAPuyF91sss96+yDHaAttdY7mcs6X4mAVkX2x5Sn+7HIbieB5LxcHBFNNO1GFQvmS6y+7g47OfMIcU9T0KnuwSnaGnsRyThLhiXNNGNEiWMK0VUQcLodlRIqyjq2ImOyARX66gBkeApLgYAPHDv/pb73D2PeyY1SIgolRp9h9CN2H3TRm53mBcYfsYylaSsD+17t91uoXnXonWIsa9ZEhnyJhzjm7stDOzC2MFBm+4bZ97XpCj09Yh1KBPkHmyzAv5n6DEC51fh2/W8JsfpG7dxoO4XKo2fdeIDtonMR3gG2i6ivVhtyDmpnvAzuOAyzifnSCDKv7mlYUY7JB4HsGw9gAIuLJg1jH8HXwchpusotdYhoGtKh2krmPji/dAw3wdjI8QtwxRzd4taYphNwfbT+LwZx+4VhkVqAG0aC39z06RsW4NzCedftyTQQkZR+bmGdUqoYcR8/3wFMAV1PSV6mMxwDiwZza1+EGbQzxQMZSXubpCw0GNkQVOOgPmLXx/CmImzb57j8JbnYbr/N/SYfZz9VTZ0rFIWBq5I39jm4zC2sI+fJtK4vEjaAJ0GK0x6ol0C08qoJQ3C7wFWfGEJG1aFJfFhiwkbhBvdjvHx0USXBoCUuACEUIQEIcG0tnDumUXCg/MCJK+AeiRdeLh7EI7uzhr3h2QkF3LqTfh6FtDKRR6ZNan9fVzngcirwbOJmN+g8lLUhtPGFIi4SIlyD9XbufbzopMh9sckttLjlsUrZoowRDa9EsbjgooTuz9Ed4zPHOz1cSZlWhiAmPe1SWZFHbIpia6ZQdQfv3xS5uvw3l5tkTs0IkcYLOj2sKM03qNbu96Mo7FklN9t/bE2RFEe572RtTQ3l8ELaUExrOqOm/3r+OF18bnAESA5+HDmZ7IBtCVzvNgu29KVv4A0+wZoQSM1Qnodgmt89s23IIbvhBZFd7Gof00xatwweGQ4eczCMOK6X/BkyhAMG1E2aH98gA/XhIvQT74g2+MbMgp1a5QU5axZvf4iY3Y/LVxws73CvayCITJubJQirtbjdHADffI3bYDeTgzqQgtZdBZmx4W+PH1Rxf6ctKfZ5ucNQrY0AhZf2tjx7yNxfPqsja5XSuAiWEmlzau7HeDhJLBH2GaBtzwBMITTsSwsSC5kdNgsPInuelWAZ527N1N88wpjQWsrCIGGgUqU8/lvGoAMkGh0ko7eVuCzPX3CRPUZq5w1Bx31PoORE4Mc6MvrW9G1wNuLGdFJZbwsk0jhfKubbHQxc4AJbbXhk5qygokaZi56lM46CjKErCweuk5Pb+4vGhvCtRqRZllMAZqOKHFhRYuMSyPfykOXGN9F4LWjjNF3fs52oEj1fBNYms037nh3H9nn6XIneGkfoKIfPHEtYOaH5YLoYT8ZAxd51IrnhdXH2TVpcA5UVL5gLr52oDNT20ZeMN5eI/i/L4f/qcv6QXFaV194QQ837w/u/fjYSnzpArANxi5iN/LypQwZVLy4Kf+fuZGu/gJQSwMEFAAAAAgAXHtiXUN3C1OPBwAA0BwAAB0AAAByYXphZWxfY2hpL2FyYzMvbmF2aWdhdGlvbi5weZ1YX2/bNhB/96dg80Qushq3XTEYdbGi78Ww7c0IBMaiHW0S5YlS2rTLd98dSVEkRTnt8pAo5N3v/vLuyKurq9+qL6Jel6KrHkRJDnxQvCaSP1Qn3letJMe2I6pqzrUgH37/SPhBr6ozPwiVX11drVbHrm1IURyHfuhEURAgbruecCnbXmMoS1Pynh9qrpRQI5FbMhSHtq6FluAoPraD7EWXkVL8MwhD1j+eK3kaKT7Ix9XKfsuhOT8Srog8r1arXx08Bb6vQu7+7AbBVnqJfHJG/nEQUmxXBH7Avrbbkn4Ag/fHuuV9RvSfW7Pd950mUSNNgjIjeZ4b+vbuL7AnIq4kkCb4LCPZEcpA+1Icifii5RVTQAqFyoI9vBFbtJ2R9fvYFvIv+dSOJj3wehAAKs85V7zr+KPhzvVvcCy4U+xgF/TavGWapzoatlyWVUNe7MhrAnlgltQ9P4v95pa8I2/i1Vd61cjFn05AUkitjF5TkBGDKrr2swKNGv6FbjICcmkHYS5pJOAncpPf/MyY1anhJ7RDE+3XwLhde3jG4QdMF8S2eUMR/CDqmulMxi+QZ7DyTmhRdL1hzuxaSGpAGJjyetmUYyXqsoCMhWQAed/0lxGCXxkpUJCByptW9UDbNK2kr9jTygA2vJKYyMDtxIAnPmcIMTSSuVWEhQ1E7Lg8CWoMMJ66uQ0JDXOaduPRgrXoHb2998TeMgJHF/l9GzXbk3UynDYpJCZ2Xal+P8tu+GWzGdN5b2Lz+b6CKuLMnlwLUYSzC/lQSeq2med5uwQBa9oHQTX9tH8WkKXajbpG0L3e9wzFqKtJj0kXyzlposVNngAeS5KfWzDu2FMW0GrknJ+RigahC+kwLFJUp/s7+ADH0mB3jDtZk40DyNIk18+SOOUB7TmS6xkJ2844IE983RMBDNwXR2vkZUny0b/Wh3NilN5iO3mkcZhQm283GQkPA9r9RIJzEBNtDNEsRL5r8PBidFcLnpkOwai7zn9dbFRcT0ZSrCmvlmuKh6mgndG/xeOu5s1dyY0uW0I1nhaR6fNixUHDQADsGdAAsPBZhgtHU3eORB+a6Rc6XhNRNTTUego9V8z8xshLMikbZdmEgWF0ri+e57aG6iYMx3M01jnOVcOpVQPZGJrJM1M/UKZOO4DN1iLY5u3YA98HWItgbOWHOWzSVGu3078zT93d9JmNOuzsX5wLzPTyUU9qFhJGFhf+ooDT1xcFVaI+muHAjQK62MJyDiJEc+5FuYX/e0wMrNPw6dU3Tfgg7LRTVofeJBBMCrLUcwTyfHsKGUzD2479d4Qe23EEr80sVPUVZhmgtHML0Lsj4Uihpp9Ev0jr7G/vlOig7ATcU/6YAbaoSg0yrd8JCCEgpycpj/8IZlwmSzjdSQV1see6/9lCXHJelikqqChGU1IpYz8kntZpXIiamU49t1RW6lzD6N5AcsZedtD5mPhclgZ7XAmx9VZx92hjMmWINxOEA26UMGPZRf7xROH5uSBzJjeHpIWw86GGeqJx9rdsrMgGMmw82JPKcBhY0iN0x1yRhveHe4EFIlQJ8nTSJdVOsZBYZobT9SbdSg+t7Cs5iNlmKeqehyO9hYPy50302u1v30BC+qRpWcJesyLeGfHcIEiqnssDHkVT14G7riSvT7lsu4ZqbVnSERuxfgst0SG825Ff8pu0N3Tg3Kw1smQ2aikhICCqMu7UYGprwLksaGURVxafG7y4aO5QnqjTEnF+mR9Nl0EYSLjoUeOosZ16ZmEumrRFfATd7WI50JJQRDI6oeog0T/zwb7vHtT60gmIgL0EM+fXNrYfTUYrcoF7lU7FBTsu1kI7WOLFVx2xZwrqY0ARqWvKsL7G+exTkffzjI0E6ZZIdn5/3LvSPlWhcyceqnZQI6VtvbqaOHLdfr+KrlX0VcK36e7tScPHBSfoJ6vaNQlNeknouLFhqfbuA15D+fKGz/u2VULPHpDC+l0l7pWZ7YZuONU9ww2kU2ufnFqLE6/dHIajsSgpTitTLzUjmH2ewh5iZDDPJYNUQuCdzqofcRghkBV2yd6Bw75866ePAUzF3m5BNY7TTfsk996RUtxBXzZn/NxWJokMP65NFP6oGwtwVHaAGoHi3n+hbulyYHDTg4DmCmrZ/Npp22qqzwbQF2vcDDYczub9yYTTaXexGIeO9thmlNaV8QUh2baob236DvysT5Jco59ezPz0HT17DLg3VV92jutWcFkvDNczhk9TRdLAyIdx1D3FkuhsySexkttF5aIDgbfpRdqU+dlFau/uriVs7XB0kQl/om7j9UqNE/VPL2kZu4jNlvVNM4paiWXvzdInKileSAJH+yNGmANpWZ6I7IdCFoIvG+8FCmaA5rvitBwjhMBX1v8TpYUIpRkS/r90ZhafQ9zzzcLJnL/E/WjviobDyRWJASbqVpNRJuQANou5/1yydOi+40xpCC84sSqFGaZCY8zWJTsOkOdVyXsRPUAnZo/QnU7cwjA463OWwXsXiKw2BPgEideu+WlLXjefu9yZCF+P0teht6Jkn5yRuMdZs1j8+qhfOR0jw8dbsNbzK9Ypk3b/AVBLAwQUAAAACABce2Jdf6/upSECAACYBgAAGwAAAHJhemFlbF9jaGkvYXJjMy9wbGFubmluZy5wea1UQY7UMBC8+xVNTrE05AEjLRICceSAuEWjyJu0F0seO9jOLhLwd9oex8kkM4KV8Cmxu6qryp1UVfURA7qzMsoH1UMvJi80BCeMV0FZA9JZExQ6kNaBNVoZhPdfPkBP287qpqoqxqjoDF0npzA57DpQ59G6AMIYG0Sk8bmmt1pjn3bmogG/T8gY67XwHr6Wzp9y4yMDWgNKakAyQ9fVHrXk8PYdfLYGL+dxxe1GPAulxaPGIwyqD60P7gBhGjW2yoQDNE1zOsED/Px9jVss+4y8gBKekKcD0OMtpAgBz2PA4UjvO1hE0HbNWfHhsLdu6DxFg8lLpKbnIyRUcdCJPuvZ6P+r9zbxxdYJWXsKGoc66tixc76XtoSR9T0iXX8RmICdIsNJkpA0QunsnrBVuG194Urp1IWK86g2Md1JtxHDUN8DrywY/BGyt124SR9B4ddGZRzuQkcVmzybJ0ou0Ryg5nzBxaUkzGeLIqDJX3jKhFwB43JIn4xZgKxUjGgGZZ4olfSF1O3cJCrnJ14KnxVFi0OczFSxjOfLN6VxJrru3U/OYbw8qZyf8yKKXNyMluZGxrHdWM1AePOQc91Z+pcsS/tdmutUS9Xrc93ku3bJ/q9YSb80+3K5qe2wJ/gtG5zd8LswKZ9uGaxbb5r5qm/bjX9kZehfuj3IqPQBFbq9gvnqxRif6ps9FvxhHZu8HqNZPmqPVyf7pstOvqkIZH8AUEsDBBQAAAAIAFx7Yl3ROaCtwAsAAHs2AAAZAAAAcmF6YWVsX2NoaS9hcmMzL3BvbGljeS5wec0b227cxvV9v2K6T6RNMbrYSkGAQQRXMQK0tuEYCQpBIGbJ2RUhLrkluYo2qv+9Zy7k3MmVnBbVg1Y7PHPmzLlfqOVy+XG9LvMSVye7pirzw0nebHe4L1cVQVef351cvf/55ALhvC+bGnWkIuyveLlcLhbrttmiLFvv+31LsgyV213T9gjXddNjCtYtFmLtDnd3VbniWwrc47zCXUe6Yc+4xCH6w66sN8PDq/owIqr3290B4Q7VO0FAvCaYEjDiChYIfnK873CVrVu8JdkasJF215Z1H7GnpM6bgvCnfMUDCDhKUvdZ3jRtUda4J120CMXJuwruqhD6pcV1V9Kb/9Q2dV+SVgACOtI+AB7KlAH6HaPwHXD9/hNFRNoIsW9fcLshfYTIY98C57OcLmY9WxUMimv8UG4Yk3V0H/h6A7g+jCC/5KQmEp/cm3X0iUD5e9NWxcjzAu/68oF8bgDXb/RJhOTf75p6XW4Wi8WPUnDsN8o+kRr4tJGsSBgfuQZlZZEgYAZbanakpZRmgwATEGpcF7ht8YHLxPdgEK6UVoK6vuUSgxtUJEGrpqnYgnndxOQM+jf60AAbGGaF14kqDhWoIg+k6jJqKhXpibjTQrCAsQls5+ITsyh+fzCYj3VVwmHsMeLWhvo73KMclGK/BQVu6uqAGmGQqFl1oDTckL7j/Ou44VGEBVmD7ZV12WcZ13h2eVKto/HbK/knVXfgyH3WlX8QRi9K0duzcwnxe1n0d8OTyzeRglPcENa/v3grHwiZgrECRjBSwSIAox8Sbt+RjNtRLuQCIF/avQEi5TQC/YSrzoASomdy6qYA17isqF/CQpMntoTo5AdGczJiKNfq9VDZGc+5ywCzwS2o0IYKVljOe/AjV2zrQoNW0aUqlCq7WIdSvulgKksBTv1qAyq+IjU4bQNr/BXw2pq9xcNpsdnzVEfDzW4AJbsmv4PtpzoQVURYpR/6A+65UofTCjQRmA4sYDqfst+RbiKp9i1ih6b0VzhiDHUidtzxJQ4XqBuGvo3syo6GontySFC/B4dyA54sou7sdmpf35akUMNSgooy728UFGCyt5TyYZEGNbZ4ewson77qCDdVswIh+/Cyzcfi6iD+g8S59ubNvu695NHfDgxrEUCZpzCjamCwvh6CHkAbYdAE5Yq24/F2BFeDsLljoCTb0ASBBjCgfkoyq6rJ74GHPBAJLsD9JfPG698yde7hSOnTOe9qOCxg3pxnJszBMj8Fm6UbagkEyJquBRCjcA+fDJxLIEJLimcpcPDFUD0MFw8YfIZIMVqyKcWxDpc4Yajbsg7OIz/Ea3QWHm2y8nmcMzMNZzQfjiTYkptPn93QDo0dASW7IBchvYCSbALJ2BIxnHn8+fqX6y8KKh7dSSbchiXryJn9uCOV6n/coYpTJa886hhTQNitqh0jwiWvWBAduAJbSf2jpCMelyMN2sr69F3WY333iqwb0Cr3XvcWvIbcW+5wA+3aZgOLXVaQqsfpumpwLyzJTPbQicZv63moY6aGQEOHwvEULd9f/eM6+/jr9edl5IgnhkhjkdI6rmUnwiBQX+ljCtZycXFLcrATsJnB4+qSNmNdbB8fTW/wKMXUhXRInXqapXn5kM7RmrhpdfvvGBeFzQ36E8yxxMOC0MJmXc6RwnmIHkOg20afKwwL1iyi3Ft8FabQvGjixqSC+9ZN/z/TfN3kYTOFswh8mQ9YzEt11Bel4KS+m7LA9t/u/GVa1sJXMs+TPtts6Q+nKvXS697F5cOPPc6oPV5Y/+recrx7nTcyX8VooaCbfAcmfltyZ1vuMmLILJW8MKfSVHKPyGiqTGQMLJ5JynYtock4LwiV8C7WA+kYRpSKoa7XJKfGxoMknFRBYlWBi2jabSBRQzDpymKPqzBUuhAPEN43JFs19Z4WiGfxKfqOUtv9q+0D+u21SpAgRO2ESOIUvCLp0u3gDWB7pdw1HvRJg3qNTuOLtwDI72U9+/6tjmQPAmx7DPw+uGH1O2owJ+jCpEnonJIEmJUABOZs1RJ8P6SIRtPLVAOXwItyQzoqM9GHjbs7fP72Mlgvn8bK+mvypKCGbyPar8uY90uDMIw5qsDiPeyhecQ2Wx0gyQ442E3yVyjxlqtyswxBzuevXl2+US4os3Lllp5elnXthZV/citQlG3APhTWY9GqtH2O5JSe0x3HtuRJknBzeqt/P7tV2CrF/6fyl7sMeegRfUIfM12u5RjhvEgIc/7JXX4Yebczz/dUH5m1HkjCQ1eKzspKmnGMYHAnX30KoTIItIxQOrGINwAkZl63Vods4ghvbUtPUry3gfpbXPf/hYsNGLBIFDl/IO8igOM0tEHPR1CDp+49R3hnjvqMnFwCHNccpw9zyzpSRBmaHv9H6u3LfEv6u6ZQ2gQTimlZFLMlaZ/SoB5HmWseifqMS7j0CHdwwZ3ZcEIZ4Ch2UHDzGKFDBMe8QvQv+DiAXypo4yMFIIby8k2ouX7a2dkz90ST5schs9FdlGDe0BEZuyPSdXAWKE7JMbS5tbslU31rXBcTza4UnRpjAK0EOar2uCeU07qaUFB+2/gBV3sS6lsGb+BxMWDpwFe8r/oAkFuWz4dmkFkWFJqiseeaLkIV35OiGso7u9QIRrwI6g15CnVWypHAePmI2iQ8ZtdwVIVyGyjpVP3PELDK3GVZ861AhWsW+wceugt/iWWCPIGP9gzpnNTG8rQUTeVlIipN8R2C+iOsadYKawd9DXKIidPVTuRMgNO72KqN6OMgahn6LcaKfcqgQC289vRDis6lL9QtiyXAzyzrxZgcdjnH5/4tg23pFXZ+1zQdCUQ7XSCxSshjynfNlPiG2KNFR+nOf0F/bB16rh7xe6mapCnIOAX2O5ng4jxCF+fThjR3b4qDXfbi/OufZBTP9PLP9PDf5t2tlPHbvd7ipVFjfBsn1V6yMcE0mW/xY+AJVZEZN9MKb1cFliDJMMDwFjv2oMK+tt34mx4YGCqRTva2JFHpSLYR1rw1hje66UL3gj3XdOZdRrh4sTGN+R93q3o3i1EDBWZVQmELOR6cXlEV6DNjAqrN2wpSiZ1HTrdUnAo3ORPohr7VYAb+2OMZczKM/pIOeBL/kGVuov3MqfbLJttHv3gxNxUxh6nesTkaeWP6SC/DI22urYKAXm72oMLwuQxDWzLKSxVUKPQD2Ehf/PGhW++rio93ASl7NSgMHULkfQg+BmY65bq60cK1ns+PpJ8/xD5qkG0Bq3xibFrM+nG3+QgXYMy0dR7brWS9jw4FwdPyw8cv2ae/X/3z+m9UuLKt/jXxh3FtKh8uDNdHb3BjX12dzbO+WlkEntEcLWvGR5RMTVvxAxgPXkmbcM4Xpo8bMkfnfQZUt2ZaNbxZonMGlx1Bn0Hu5ZZcty04iyWnl77Py10lbW4CBlSRDa7gj/qEHTkgXIZHZz0ebWA+gr6/VlWBpDQuhxHZYxAqXKUsFUdrGkXynmUzmhU9b+7nps8cTyrFgjW51DAwcfLLsR6SVTY5xpXOCTv3HY7Cd35QxhoeduLm4edU9cxjRSE0jbriudmyqIactchxE2JOvfMR46F5LS+k87p+vGvap7An8zNNF7PSHfqv7mDo3BpGR5RY7lE/K7ctXVWKTAuPrZTOiacheOcbQ04F0XWZ9oMGN2Uz1wo3gzl72kjiEKdcKdG2xqepQaGD1/qbyK7XbDhZ7remcH3P66mmBSBfReEoDZRCZehgupX+xG8LalkjuOwo1+yoav6IFrtzzOrKFuxphW0lEToNp07UWuTmHPMlFhh63jXgO+vx3zgm9ngVkosZyh2jghBs8ZXyAwaLbh9nb0y0t4NJ+cRgbqBsp9xdeOdZ1L528R+kbbrgjasFP4Ya9b2T1NUA5JdTY3YyPUuLTKTDhGSu1W/KRC/8PEI0Um37rWvfW4oO2c0NCWfeSrR6BXHe7A6BgdZ2kelcvOfxJRWpoS0QHdoMFel0SFaFlfpf6zHfbUrNvGrqDUhXr4BdY/EfUEsDBBQAAAAIAFx7Yl1FJjXNoQkAANArAAAYAAAAcmF6YWVsX2NoaS9hcmMzL3dvcmxkLnB57RpNc9u49e5fgWqmM6QtMZI32baaKBPPbraTQzcZT9oeNBoOIoIya4pkQcqxst3/vg94IAgQoCg77bSH+mIJeF943w/QZDL5UORZwQjdNllZzLZlkWTiE0vIbdmUnGzLhH0uy3vypeR5QuqGNoyksHFz+8Ps5s/vZ99Fk8nk4iLl5Z7EcXpoDpzFMcn2VckbQouiBBSgWCuYhDZ0m9O6ZnULpJcuLtRKcdhXR0JrUlSIJRei5lhlxa5F+/nHG87pUdGNUkYFb031p3c3n/56+y7+8f1fLi4uPnx8d3vz6cNtbCyTFXkJWwlLSVyUfE/z7CsLHmh+YMuW+rqoojQvafP9y01IZm9868sLAn+CAlCUiwHsgl5pvovEMtIMQwnHGYhZELlEXiBaluL/N2TBZotrwvKaIUS0LatjEIKYb7WaAjjwV1asPnEgeiGX0Fp/Fzb6oSzSbIcyfcmS5m5JsqIByb5/Kddai8Y1nLbde7W4lpvoB3W7/Ae5+Dkvt/d6Dak0ZXXfrvwRoVhDl3h8sRbN5eqe8R0DVvsspzxrjh3EPPoTki8rxilIHyNskoGTFVtmQi5e2ZCgQ54xUJAX+jtknTPKC/CXGHDsfaRWM5bog8KaXJTOEFdl3cRZkTVxHNQsT9HyEBaoVvEHJhM7kVQxeU2uCQSFsfJ7ct0BS7vTDIz6N2HUd5yXPJhghGFgIdL+UDegSAgbwh5YIYRjO8bJmxW5noQOb8uUIMNijKON0HKD00LQPzAPB+UPgrY+H7rDOexabFokyolGeEKyAEavV8hI+Jj+YrvtCGOJ2bL6kjV3WdFlsi2t6BZc0XPcAVeEs86j+RjTIeRWjgISbMF2dEDVJ5wb+J8S71zBiKZMbOG25QM4WXPHMFz17uTMvPORsySTtlapkD02cZuPvalUZcI6Sw40H4aoeLkDqFpFr1xLaZYDXXPpALLyhkK0HK3lmiWxMDvmLjgLynyT0EoYocuZKDYUsvd1uRdK2pI70EHJj6riVfmhJqWqlXVdbjNpRtJwWtSyYmr/qrEgdsnEyCNTgJLZ2cnXngyj3F7sQobCD3pzxwpl1BUUyQjESMp9BAzpIW9iXuwCRIhElgttmqDVfzBpLkDWhCIsgYHlTOD2U7KI5qJQVVH9T94ERgENp0RE48pca4+IabDj3JNBu/IThbEWbMF8JT60EVBaH2BP7OmQ3LTY5dBjmKIeiiwVRd7iNAOpqgyku7YlGFhGwZQIKm1qkTB12hKSF0BjWMx7dqzRM74yXtbBEGkrq/Z1QBJouNiqC8ohG/a5WUc7k7Xfeo4IQwduc8k3y/E89m2ierLOx7Ws8t1/gvS2PBTNtxGGvOqQxZRpUDXdalQoib2eb4AAxHeXSEEFrInl7lBP1uFHaZbnASSMM4nHeGDI1Ql7VKm6XcImUbKD/0tvvwIbgYYPjabF1OZInU4nh6I+VGJ4gfEL5islARES/KKp/2r0DmqW6HGfkYVxMl5qlU2xmnnrbXdeoYLuyIPzjmzID011EP0zKl3NKu2uGBNl7gJicNxih1JEVloLba1AStkfcGIEsmKOoi4m5idyhdRD6LWdXYtoVkB/UyuXpHxXg4oDg1Noi0Az6GcAGE+3NgA3FmDO0mZKeLa7EypAtPV8ubwGZapvC/HNQtpC41soUeBzYFSVtWmBKR5uE/bqRIsLn56Ii46QIDrbV80xzrN7FqCoXlg8jew+JOdLeWZwMPVNnt2LuFCIJtpVR8ZF1K7UUlBG2/SdHQENBwezN9BwBlakd5XRcetuK+V0z043qh2wrnYn4cm/ZE5CtPEAghxiCxHVd7Ri5Hfg+2Z3Eo7mDpsM9vV3FFpUJBj8YpD7dRraA4hzNpLVvdzqVYGZ5P1d1XCy97E1Tu8lN6YGa1daeOLK7KrGx0voyCLXSS5Gc+GpxqVRT/tv+4126LiRje9K+XawT3bSv0FH4pgpHyuendxF5sQzXGlxwi6gKpzmxuIp+ZZgunxmXIHWeuHlHUD7cU9UPbbLfFcyNZIYGdGp05w2MLEL3w6MZsnKsxvyhswtjxb4kbxcWa1I/8YArdUT2PVZa3pe9dIDltipg9T2vysdj6fD0KXQtrArZ8wypm7/pjF/rxZeAD2Jr3q7//U0pKqH9pC2mtj11LbC1BXBDXB9PaPEM++D7ZruzFK9Yi60t4Gi6zC1VUkfs3q18I2E0AzQXDQ0rmi9TvXEJVS/7xWtVHEMJO3w/57uerroynU+CIfcXmUc8W8tdbk57UnuYh9NX7ZnEkHfB3i96q0OgJmHtBElB1ny9m3XqfxFXLROu6xnTVtQrUAXJU8YtztvU8BwPZO0lxsHtdWKTWpjePUua/QJ26GCNRT6S5PFGIHZSn2K9vTRGF++MNGkquhlj1WAUC7Ai1X7MaoP+8Csz+iqRAO0ZV1fV/SM0oraCWncLODTUo9Suz9EyBjH0LGHCLX3DOOEtJ9InSdM3CO3RPuXXds8q9zAFxd3Pn+LhH+F5MW5OcmNP2/UOpHqu5bUxrpc+U7oGmQErlX3CFjBKFBrjCcyrcsTTryeLTYbsyPtmWGoYzHsib5qJKIutZnPmNKA8+gVBFWAZnMlFg2lfRcrvl1pscKpvkg2GFa6Mgh+dgN9pc3hKTtZnYo7dZhcNYUwonkeeMeDn8RJsmL3sYSjqRcR8RSDNIj5CNfRc+9Y+sWsg51qWafaOaat+aemXqfErglG011+rhl/YENN9+V4//38Phpfb9OSn92+07Rh/FzgVidxwnL9RDy14kQ8xH4uy9wcm62271nN/P90R9lT+MmW8pumkjovG3z/8s0py97llBQ1fl4L4VfpuW3wv7MV9rXDdrY3Uy++Rcs0JOITOxXRC3jV4atboZewoiebmD6v/r0eGEncd3paZInnrIMNfdK5fMRvB8xq2v9Bhqu3IvE1mS7hodHBfpx26MvftswWQ3crsLv0qcbAEC/mYvF1f8KWl5pPCRQxpHsNIewmqa3nm1AwlF9wspcHMByla4ldDj3HGAokwRWvWd1AGo2JFtmJCZfMYNupSNh5HVy9l6VcikPtpyKIbYRdATwqGeo9LSpYKDzY/j5H4S5OuBbHdyrTi60fDTkNzQi7pxnbuLZTnZXgGYrp5STmFcp92XpL+Dx3cXONT4pRSg4VLd5pfwyf7p3nSXySyrC0wZjvh2Pij4TCedKfJKKFtQNqTLCR6DpPsJNEtGDDkfq0uL2yAte9b+8Z68R9u/Ui/eybcqNTV/vYsONPW898MH5WE2s88Q5WmXaU+w1QSwECFAMUAAAACABce2JdGwRLWCsAAAAtAAAAFgAAAAAAAAAAAAAApIEAAAAAcmF6YWVsX2NoaS9fX2luaXRfXy5weVBLAQIUAxQAAAAIAFx7Yl1DQOWIowAAACIBAAAbAAAAAAAAAAAAAACkgV8AAAByYXphZWxfY2hpL2FyYzMvX19pbml0X18ucHlQSwECFAMUAAAACABce2Jd1LxS6GEHAACNFgAAGwAAAAAAAAAAAAAApIE7AQAAcmF6YWVsX2NoaS9hcmMzL2ZlYXR1cmVzLnB5UEsBAhQDFAAAAAgAXHtiXWshIoodCAAA2hoAACAAAAAAAAAAAAAAAKSB1QgAAHJhemFlbF9jaGkvYXJjMy9pbnRlcnZlbnRpb25zLnB5UEsBAhQDFAAAAAgAXHtiXUN3C1OPBwAA0BwAAB0AAAAAAAAAAAAAAKSBMBEAAHJhemFlbF9jaGkvYXJjMy9uYXZpZ2F0aW9uLnB5UEsBAhQDFAAAAAgAXHtiXX+v7qUhAgAAmAYAABsAAAAAAAAAAAAAAKSB+hgAAHJhemFlbF9jaGkvYXJjMy9wbGFubmluZy5weVBLAQIUAxQAAAAIAFx7Yl3ROaCtwAsAAHs2AAAZAAAAAAAAAAAAAACkgVQbAAByYXphZWxfY2hpL2FyYzMvcG9saWN5LnB5UEsBAhQDFAAAAAgAXHtiXUUmNc2hCQAA0CsAABgAAAAAAAAAAAAAAKSBSycAAHJhemFlbF9jaGkvYXJjMy93b3JsZC5weVBLBQYAAAAACAAIAEUCAAAiMQAAAAA='
archive_bytes=base64.b64decode(FARSEER_PAYLOAD)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as stored:
    for member in stored.namelist():
        target=RUNTIME/member
        if target.resolve().is_relative_to(RUNTIME.resolve()):
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(stored.read(member))
        else:raise RuntimeError('Invalid embedded runtime path')
sys.path.insert(0,str(RUNTIME))
print('Embedded FARSEER runtime verified:',len(archive_bytes),'bytes, sha256:',hashlib.sha256(archive_bytes).hexdigest())


In [ ]:
%%writefile /tmp/dcie_arc3_runtime/dcie_cognition.py
"""ARC-AGI-3: online dual-model cognition, research implementation.

Causal computation proposes transitions; Survival State Mechanics-inspired core
maintains independently learned risk and contextual survival morph evidence.
Neither neural model is pretrained. A/X/U means witnessed nonterminal, certified
invalid by official action contract, or unknown; observed risk never becomes X.
Original theory: Joshua Matthew Carter.
"""
from __future__ import annotations
import hashlib, math, os, random, time
from collections import Counter, OrderedDict, deque
from dataclasses import dataclass
from typing import Any
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

# Inference and training occur inside the official action step; keep them bounded.
try:
    torch.set_num_threads(min(2, max(1, torch.get_num_threads())))
except RuntimeError:
    pass

DIM=64
ACTION_DIM=13  # 9 one-hot action slots and 4 coordinate features
HIDDEN=48


def _tensor(x, device):
    return torch.as_tensor(x, dtype=torch.float32, device=device)


def make_action_vector(action_id:int, coord:tuple[int,int]|None=None)->np.ndarray:
    vec=np.zeros(ACTION_DIM, dtype=np.float32)
    vec[min(max(int(action_id),0),8)]=1.0
    if coord is not None:
        x,y=coord; vec[9:]=[x/63.0,y/63.0,(x/63.0)**2,(y/63.0)**2]
    return vec


def state_hash(frame:Any)->str:
    raw=np.asarray(frame.frame, dtype=np.int16)
    return hashlib.blake2b(raw.tobytes()+str(raw.shape).encode(), digest_size=12).hexdigest()


class RecursiveExperts(nn.Module):
    """State-dependent mixture of two recurrent expert transition cells.

    Contains real trainable weights, independent per model. The recursive
    refinement is repeated at inference and trained at test time.
    """
    def __init__(self, out_dim:int, *, steps:int=3):
        super().__init__()
        self.steps=steps
        self.encoder=nn.Linear(DIM+ACTION_DIM, HIDDEN)
        self.recur=nn.GRUCell(HIDDEN, HIDDEN)
        self.gate=nn.Linear(HIDDEN, 2)
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(HIDDEN,HIDDEN),nn.Tanh(),nn.Linear(HIDDEN,out_dim)) for _ in range(2)])
        self.memory_cell=nn.GRUCell(DIM+ACTION_DIM, HIDDEN)
    def forward(self, x:torch.Tensor, a:torch.Tensor, history:torch.Tensor):
        h=history
        seed=torch.tanh(self.encoder(torch.cat((x,a),dim=-1)))
        for _ in range(self.steps):
            h=self.recur(seed,h)
        weights=torch.softmax(self.gate(h),dim=-1)
        o=torch.stack([expert(h) for expert in self.experts],dim=1)
        return (o*weights.unsqueeze(-1)).sum(1), h
    def observe_hidden(self, x:torch.Tensor, a:torch.Tensor, history:torch.Tensor):
        return self.memory_cell(torch.cat((x,a),dim=-1),history)


@dataclass(frozen=True)
class Proposal:
    action:int
    coord:tuple[int,int]|None
    @property
    def key(self):return (self.action,self.coord)


class CausalCore:
    """Recursive, trainable causal-state transition and progress model."""
    def __init__(self, device):
        self.device=device
        self.net=RecursiveExperts(66).to(device)  # Δ64, level progress, observable change
        self.optim=torch.optim.Adam(self.net.parameters(),lr=0.0015)
        self.h=torch.zeros((1,HIDDEN),device=device)
        self.replay=deque(maxlen=128)
        self.n=0
        self.action_counts=Counter()
        self.delta_by_action={}
    def predict(self, features, proposal, h=None):
        x=_tensor(features,self.device).reshape(1,DIM)
        a=_tensor(make_action_vector(proposal.action,proposal.coord),self.device).reshape(1,ACTION_DIM)
        h=self.h if h is None else h
        with torch.no_grad():
            v,hnew=self.net(x,a,h)
            delta=0.24*torch.tanh(v[0,:DIM]); progress=torch.sigmoid(v[0,DIM]); change=torch.sigmoid(v[0,DIM+1])
            n=self.action_counts[proposal.action]
            empirical=self.delta_by_action.get(proposal.action)
            if empirical is not None and n<10:
                delta=(n/(n+8))*delta+(8/(n+8))*_tensor(empirical,self.device)
            nxt=torch.clamp(x[0]+delta, 0, 1)
            return nxt.detach().cpu().numpy(),float(progress.item()),float(change.item()),hnew.detach()
    def observe(self,before,proposal,after,level_delta,changed):
        x=np.asarray(before,dtype=np.float32); y=np.asarray(after,dtype=np.float32)
        av=make_action_vector(proposal.action,proposal.coord)
        self.replay.append((x,av,y,float(level_delta>0),float(changed),self.h.detach().cpu().numpy()[0].copy()))
        self.n+=1; self.action_counts[proposal.action]+=1
        n=self.action_counts[proposal.action]
        self.delta_by_action[proposal.action]=self.delta_by_action.get(proposal.action,np.zeros(DIM,dtype=np.float32))*(n-1)/n+(y-x)/n
        with torch.no_grad():
            self.h=self.net.observe_hidden(_tensor(x,self.device).reshape(1,DIM),_tensor(av,self.device).reshape(1,ACTION_DIM),self.h).detach()
        self.train_step()
    def train_step(self):
        batch=random.sample(list(self.replay),min(8,len(self.replay)))
        x,a,y,p,c,h=map(lambda pieces: np.stack(pieces),zip(*batch))
        x,a,y,p,c,h=[_tensor(t,self.device) for t in (x,a,y,p,c,h)]
        self.net.train(); pred,_=self.net(x,a,h)
        delta=.24*torch.tanh(pred[:,:DIM]); target=torch.clamp(x+delta,0,1)
        loss=F.mse_loss(target,y)+.2*F.binary_cross_entropy_with_logits(pred[:,DIM],p)+.2*F.binary_cross_entropy_with_logits(pred[:,DIM+1],c)
        self.optim.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(self.net.parameters(),1.0);self.optim.step()
        return float(loss.detach().item())
    def reset_temporal_state(self):self.h=torch.zeros_like(self.h)


class SurvivalCore:
    """Independent recursive survival model with bounded contextual κ-morphs.

    Witnessed A means only an observed nonterminal transition, not guaranteed
    future success. Hard X is restricted to official illegal action contracts.
    Nonterminal/death evidence is probabilistic/soft, not an impossibility proof.
    """
    def __init__(self, device):
        self.device=device
        self.net=RecursiveExperts(2).to(device)  # death / stagnation logits
        self.optim=torch.optim.Adam(self.net.parameters(),lr=.0015)
        self.h=torch.zeros((1,HIDDEN),device=device)
        self.replay=deque(maxlen=128)
        self.rows=OrderedDict() # (current observation, incoming history, action) -> outcome counts
        self.context=deque(maxlen=3)
        self.n=0
    def history_key(self):return tuple(self.context)
    def _row_key(self, fingerprint, proposal, history):return (fingerprint,tuple(history),proposal.key)
    def observe(self,before,proposal,after,dead,changed,progress,previous_hash,next_hash,history):
        a=make_action_vector(proposal.action,proposal.coord)
        self.replay.append((np.asarray(before,dtype=np.float32),a,float(dead),float(not changed),self.h.detach().cpu().numpy()[0].copy()))
        key=self._row_key(previous_hash,proposal,history)
        row=self.rows.setdefault(key,Counter())
        row['death' if dead else ('progress' if progress else ('changed' if changed else 'noop'))]+=1
        self.rows.move_to_end(key)
        while len(self.rows)>4096:self.rows.popitem(last=False)
        event='D' if dead else 'P' if progress else 'C' if changed else 'N'
        self.context.append((proposal.action,event,next_hash[:8]))
        with torch.no_grad():
            self.h=self.net.observe_hidden(_tensor(before,self.device).reshape(1,DIM),_tensor(a,self.device).reshape(1,ACTION_DIM),self.h).detach()
        self.n+=1;self.train_step()
    def predict(self,features,proposal,fingerprint,history=None,h=None,legal=True):
        if not legal:
            return {'status':'X','risk':1.0,'stagnation':1.0,'observations':0,'next_h':self.h}
        history=self.history_key() if history is None else history
        row=self.rows.get(self._row_key(fingerprint,proposal,history),Counter())
        total=sum(row.values());n_death=row['death'];n_ok=total-n_death
        # Observed successful execution is A, but never a proof of goal reachability.
        status='A' if n_ok>0 else 'U'
        x=_tensor(features,self.device).reshape(1,DIM)
        a=_tensor(make_action_vector(proposal.action,proposal.coord),self.device).reshape(1,ACTION_DIM)
        with torch.no_grad():
            logits,hnew=self.net(x,a,self.h if h is None else h)
            learned_risk=float(torch.sigmoid(logits[0,0]).item())
            learned_stall=float(torch.sigmoid(logits[0,1]).item())
        # Calibrate: no training data = unknown, not 'unsafe'.
        count_weight=min(1.,self.n/24.)
        empirical_risk=(n_death+1)/(total+2) if total else .5
        contextual_weight=total/(total+2)
        risk=(1-contextual_weight)*(count_weight*learned_risk+(1-count_weight)*.5)+contextual_weight*empirical_risk
        stagnation=(row['noop']+1)/(total+2) if total else (.5 if self.n<6 else learned_stall)
        return {'status':status,'risk':float(risk),'stagnation':float(stagnation),'observations':total,'next_h':hnew.detach()}
    def train_step(self):
        batch=random.sample(list(self.replay),min(8,len(self.replay)))
        x,a,d,s,h=map(lambda pieces:np.stack(pieces),zip(*batch))
        x,a,d,s,h=[_tensor(t,self.device) for t in (x,a,d,s,h)]
        self.net.train();logits,_=self.net(x,a,h)
        loss=F.binary_cross_entropy_with_logits(logits,torch.stack((d,s),dim=-1))
        self.optim.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(self.net.parameters(),1.0);self.optim.step()
        return float(loss.detach().item())
    def reset_temporal_state(self):
        self.h=torch.zeros_like(self.h)
        self.context.clear()


class DCIE:
    """Independent recursive causal/survival minds + bounded pre-action rollout.
    All action choices are made before one official env step, then observed.
    """
    def __init__(self,seed=0):
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.seed=seed;random.seed(seed);torch.manual_seed(seed)
        self.causal=CausalCore(self.device)
        self.survival=SurvivalCore(self.device)
        self.pending=None
        self.visited=Counter()
        self.last_fingerprint=None
        self.last_level=0
        self.decision_count=0
        self.max_nodes=40
    def _finger(self,frame):return state_hash(frame)
    def observe(self,frame,features):
        current=self._finger(frame)
        if self.pending is not None:
            b,proposal,old_hash,old_level,history=self.pending
            progress=int(frame.levels_completed)>old_level
            changed=current!=old_hash
            dead=str(getattr(frame.state,'name',frame.state))=='GAME_OVER'
            self.causal.observe(b,proposal,features,int(frame.levels_completed)-old_level,changed)
            self.survival.observe(b,proposal,features,dead,changed,progress,old_hash,current,history)
            self.pending=None
        self.last_fingerprint=current
        self.last_level=int(frame.levels_completed)
    def reset_episode(self):
        self.causal.reset_temporal_state()
        self.survival.reset_temporal_state()
        self.pending=None
    def _score(self,p,features,fingerprint,base,history,hc=None,hs=None,depth=0):
        prediction,progress,change,next_hc=self.causal.predict(features,p,hc)
        survival=self.survival.predict(features,p,fingerprint,history,h=hs)
        # Deliberately do not equate unverified non-survival with certified impossibility.
        if survival['status']=='X':return -1e8,prediction,next_hc,survival['next_h']
        visits=self.visited[(fingerprint,p.key)]
        acts=self.causal.action_counts[p.action]
        novelty=1/math.sqrt(1+acts)
        baseline_bonus=(2.7 if self.causal.n<8 else .85) if p.key==base.key else 0.
        # TTT reward estimates are unreliable until observations accumulate.
        trust=min(1.,self.causal.n/25.)
        score=(baseline_bonus+trust*(2.0*progress+1.0*change)-3.6*(survival['risk']-.5)
               -1.1*(survival['stagnation']-.5)+.65*novelty-.9*math.log1p(visits))
        return score,prediction,next_hc,survival['next_h']
    def choose(self,frame,features,base,proposals):
        fingerprint=self._finger(frame)
        history=self.survival.history_key()
        if not proposals:raise ValueError('no candidate actions')
        results=[]
        for p in proposals[:24]:
            score,pred,hc,hs=self._score(p,features,fingerprint,base,history)
            results.append((score,p,pred,hc,hs))
        results.sort(key=lambda r:(-r[0],r[1].action,str(r[1].coord)))
        # At sufficient evidence, expand top candidate futures for a depth-2
        # imagined experiment; no simulated action is submitted to the gateway.
        if self.causal.n>=10:
            rescored=[]
            future=proposals[:min(6,len(proposals))]
            for score,p,pred,hc,hs in results[:min(5,len(results))]:
                future_best=-1e9
                for q in future:
                    s,_,_,_=self._score(q,pred,'predicted:'+fingerprint,base,history,hc,hs,depth=1)
                    future_best=max(future_best,s)
                rescored.append((score+.35*future_best,p))
            best=max(rescored,key=lambda v:(v[0],-v[1].action))[1]
        else:best=results[0][1]
        self.visited[(fingerprint,best.key)]+=1
        if len(self.visited)>6000:
            self.visited.clear()
        self.pending=(np.asarray(features,dtype=np.float32).copy(),best,fingerprint,int(frame.levels_completed),history)
        self.decision_count+=1
        return best


In [ ]:
%%writefile /tmp/dcie_arc3_runtime/my_agent.py
"""Official ARC-AGI-3 Agent: dual recursive online cognitive inference + FARSEER.

Survival State Mechanics: original theory by Joshua Matthew Carter, 2026.
This executable is a bounded operational implementation, not a claimed proof
of equivalence to the full theory, nor pretrained general intelligence.
"""
from __future__ import annotations
import math, os, time
from typing import Any
from agents.agent import Agent
from arcengine import GameAction, GameState
from razael_chi.arc3.policy import RotorARC3Policy
from razael_chi.arc3.features import encode_frame, salient_coordinates
from dcie_cognition import DCIE, Proposal


def _is_complex(action):
    flag=getattr(action,'is_complex',False)
    return bool(flag() if callable(flag) else flag)


def _action_coordinates(action):
    try:
        data=action.action_data.model_dump(exclude_none=True)
        if isinstance(data.get('x'),int) and isinstance(data.get('y'),int):
            return (data['x'],data['y'])
    except Exception:pass
    return None


class MyAgent(Agent):
    MAX_ACTIONS=500
    def __init__(self,*args:Any,**kwargs:Any):
        super().__init__(*args,**kwargs)
        self.MAX_ACTIONS=int(os.getenv('DCIE_MAX_ACTIONS','500'))
        self.time_limit=float(os.getenv('DCIE_GAME_SECONDS','900'))
        self.started=time.monotonic()
        self.global_deadline=float(os.getenv('DCIE_DEADLINE','1e100'))
        self.seed=int(os.getenv('DCIE_SEED','17'))
        self.policy=RotorARC3Policy(codebook_size=256,width=64,seed=self.seed,
            action_type=GameAction,use_salience=True,use_navigation=True,
            use_causal_clicks=True,use_failure_adaptive_clicks=True)
        self.mind=DCIE(seed=self.seed)
        self.last_guid=None
        self._last_observation_token=None
        self._last_state_name=None
        self.reasoning_count=0
        print(f'[DCIE/SSM] game={self.game_id} model_device={self.mind.device} '
              f'causal_params={sum(x.numel() for x in self.mind.causal.net.parameters())} '
              f'survival_params={sum(x.numel() for x in self.mind.survival.net.parameters())}',flush=True)
    def is_done(self,frames:list,latest_frame:Any)->bool:
        return (latest_frame.state == GameState.WIN or
            self.action_counter>=self.MAX_ACTIONS or
            time.monotonic()-self.started>=self.time_limit or
            time.monotonic()>=self.global_deadline)
    def _make_candidates(self,frame,base):
        legal=[]
        for id_ in frame.available_actions:
            try:
                act=GameAction.from_id(int(id_))
            except Exception:continue
            if act is GameAction.RESET:continue
            legal.append(act)
        proposals=[]
        for act in legal:
            if not _is_complex(act):
                proposals.append(Proposal(int(act.value),None))
            else:
                # The official contract requires x/y in [0,63] for complex actions.
                # Preserve baseline causal click and add genuinely distinct salient probes.
                coords=[]
                if act is base:
                    existing=_action_coordinates(base)
                    if existing:coords.append(existing)
                try:coords.extend(salient_coordinates(frame)[:8])
                except Exception:coords.append((32,32))
                n=0
                for x,y in dict.fromkeys(coords):
                    x=int(max(0,min(63,x)));y=int(max(0,min(63,y)))
                    proposals.append(Proposal(int(act.value),(x,y)))
                    n+=1
                    if n>=5:break
        # Baseline must always remain eligible even if candidate lists are clipped.
        baseline=Proposal(int(base.value),_action_coordinates(base) if _is_complex(base) else None)
        if baseline not in proposals:proposals.insert(0,baseline)
        return baseline,proposals
    def choose_action(self,frames:list,latest_frame:Any)->GameAction:
        # The pending real transition is observed once per official frame;
        # training uses real experiences only, never invented rollout labels.
        name=str(getattr(latest_frame.state,'name',latest_frame.state))
        if getattr(latest_frame,'full_reset',False):
            self.mind.reset_episode()
        features=encode_frame(latest_frame).astype('float32')
        self.mind.observe(latest_frame,features)
        base=self.policy.choose_action(frames,latest_frame)
        if name in ('GAME_OVER','NOT_PLAYED'):
            self.mind.reset_episode()
            return GameAction.RESET
        baseline,candidates=self._make_candidates(latest_frame,base)
        if not candidates:return base
        selected=self.mind.choose(latest_frame,features,baseline,candidates)
        action=GameAction.from_id(selected.action)
        if _is_complex(action):
            if selected.coord is None:
                raise ValueError('complex action missing coordinate')
            x,y=selected.coord
            action.set_data({'game_id':latest_frame.game_id,'x':x,'y':y})
        # The FARSEER navigator/controller must learn from the action ACTUALLY
        # issued, rather than accidentally crediting its own alternative.
        pending=self.policy.pending
        if pending is not None:
            pending.action_id=selected.action
            pending.simple=not _is_complex(action)
            if selected.coord is not None:
                pending.operator_features=self.policy._operator_features(selected.coord)
                pending.click_target=None
            else:
                import numpy as np
                pending.operator_features=np.zeros(4,dtype=np.float64)
                pending.click_target=None
        # The underlying official framework transports the reasoning object
        # to the competition gateway with the REAL action.
        report={'system':'DCIE/SSM dual recursive TTT',
                'causal_updates':self.mind.causal.n,
                'survival_updates':self.mind.survival.n,
                'kappa_context_rows':len(self.mind.survival.rows),
                'imagined_rollout_depth':2 if self.mind.causal.n>=10 else 1,
                'selected_action':selected.action}
        action.reasoning=report
        self.reasoning_count+=1
        if self.reasoning_count%30==0:
            print(f'[DCIE/SSM] game={self.game_id} action={self.reasoning_count} '
                  f'causal_training={self.mind.causal.n} survival_training={self.mind.survival.n} '
                  f'survival_rows={len(self.mind.survival.rows)}',flush=True)
        return action


In [ ]:
import os, shutil, time, json, subprocess, sys, urllib.request
from pathlib import Path
OUTPUT_ROOT=Path('/kaggle/working');OUTPUT_ROOT.mkdir(exist_ok=True,parents=True)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Real competition rerun: never read hidden game internals, only the official gateway.
    url='http://gateway:8001/api/games'
    deadline=time.monotonic()+600
    while True:
        try:
            with urllib.request.urlopen(url,timeout=12) as response:
                games=json.loads(response.read())
            if not games:raise RuntimeError('empty real competition game list')
            print('OFFICIAL gateway assigned',len(games),'games')
            break
        except Exception as exc:
            if time.monotonic()>=deadline:raise RuntimeError('ARC-3 gateway did not initialize') from exc
            time.sleep(5)
    framework=OUTPUT_ROOT/'ARC-AGI-3-Agents'
    shutil.copytree(FRAMEWORK_SOURCE,framework,dirs_exist_ok=True,ignore=shutil.ignore_patterns('.git','.venv','__pycache__'))
    templates=framework/'agents'/'templates';templates.mkdir(parents=True,exist_ok=True)
    shutil.copy2(RUNTIME/'my_agent.py',templates/'my_agent.py')
    (framework/'agents'/'__init__.py').write_text(
        'from typing import Type\n'
        'from dotenv import load_dotenv\n'
        'from .agent import Agent, Playback\n'
        'from .swarm import Swarm\n'
        'from .templates.my_agent import MyAgent\n'
        'load_dotenv()\n'
        'AVAILABLE_AGENTS: dict[str,Type[Agent]] = {"myagent": MyAgent}\n'
    )
    (framework/'.env').write_text(
        'SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\n'
        'ARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\n'
        'RECORDINGS_DIR=/tmp/dcie_arc3_recordings\n'
    )
    env=os.environ.copy()
    env.update({'PYTHONPATH':str(RUNTIME)+os.pathsep+str(framework),
                'MPLBACKEND':'agg','DCIE_MAX_ACTIONS':'500',
                'DCIE_GAME_SECONDS':'900','DCIE_DEADLINE':str(time.monotonic()+29700)})
    print('Launching real dual-recursive online DCIE agent against official ARC-3 gateway',flush=True)
    outcome=subprocess.run([sys.executable,'main.py','--agent','myagent'],cwd=str(framework),env=env,timeout=30300)
    if outcome.returncode!=0:raise RuntimeError(f'Official ARC-3 agent failed: {outcome.returncode}')
    print('ARC-AGI-3 official agent process finished; competition gateway owns submission.parquet')
else:
    # Official starter contract: Save & Run All creates placeholder only;
    # Submit to Competition triggers the genuine game evaluation gateway.
    import pandas as pd
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(OUTPUT_ROOT/'submission.parquet',index=False)
    print('COMMIT ONLY: placeholder submission.parquet. Submit to Competition to run actual ARC-3 games.')

    # ---- PRIVATE EVALUATION (added): play the 25 PUBLIC games offline with the official scorecard ----
    import threading, logging
    env_dir = COMPETITION_ROOT / 'environment_files'
    fw = OUTPUT_ROOT / 'ARC-AGI-3-Agents'
    shutil.copytree(FRAMEWORK_SOURCE, fw, dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git','.venv','__pycache__'))
    (fw/'agents'/'templates').mkdir(parents=True, exist_ok=True)
    shutil.copy2(RUNTIME/'my_agent.py', fw/'agents'/'templates'/'my_agent.py')
    (fw/'agents'/'__init__.py').write_text('from typing import Type\nfrom .agent import Agent, Playback\nfrom .templates.my_agent import MyAgent\nAVAILABLE_AGENTS: dict[str,Type[Agent]] = {"myagent": MyAgent}\n')
    sys.path[:0] = [str(RUNTIME), str(fw)]; os.chdir(fw)
    os.environ.update(DCIE_MAX_ACTIONS='500', DCIE_GAME_SECONDS='900', DCIE_DEADLINE=str(time.monotonic()+7.5*3600), RECORDINGS_DIR='/tmp/rec')
    logging.basicConfig(level=logging.WARNING)
    from arc_agi import Arcade, OperationMode
    from agents import AVAILABLE_AGENTS
    A = AVAILABLE_AGENTS['myagent']
    arc = Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=str(env_dir))
    games = sorted(str(getattr(g,'game_id',g)) for g in arc.get_environments()); print('public games:', len(games), flush=True)
    card = arc.open_scorecard(tags=['public25-private-eval']); per = {}
    def play(g):
        a = A(card_id=card, game_id=g, agent_name='myagent', ROOT_URL='http://localhost', record=False, arc_env=arc.make(g, scorecard_id=card))
        try: a.main(); err=None
        except Exception as e: err=f'{type(e).__name__}: {e}'
        per[g] = dict(levels=a.levels_completed, actions=a.action_counter, error=err); print(g, per[g], flush=True)
    th=[threading.Thread(target=play,args=(g,)) for g in games]; [t.start() for t in th]; [t.join() for t in th]
    sc = arc.close_scorecard(card)
    res = dict(per_game=per, total_levels=sum(v['levels'] for v in per.values()), scorecard=json.loads(sc.model_dump_json()) if sc else None)
    json.dump(res, open('/kaggle/working/public25_eval.json','w'), indent=1)
    print('*** PUBLIC-25 TOTAL levels:', res['total_levels'], '| scorecard score:', res['scorecard']['score'] if res['scorecard'] else None, flush=True)
